In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from helpers.data_management import (DatasetConfig,
    create_LOSO_dataset_dataloader)
from helpers.constants import *
from helpers.visualization import (inspect_knee_moment_dataset, plot_loss, 
                                   prediction_overlay)
from helpers.modules import KneeCNN, KneeTCN, KneeLSTM
from helpers.running import train_model, evaluate_model, loso_cross_validation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

batch_size = 32
num_epoches = 5

# Dataset and Network Example

In [ ]:
leave_one_out_subject = SUBJECTS[2]

cfg = DatasetConfig(tasks = PERIODIC_TASK_PREFIXES)
train_dataset, test_dataset, train_dataloader, test_dataloader = \
    create_LOSO_dataset_dataloader(
        leave_one_out_subject, dataset_cfg=cfg, batch_size=batch_size)

print(len(train_dataset))
print(len(test_dataset))

iterator = iter(train_dataset) # only for visualization

In [ ]:
# visualize dataset
x,y = next(iterator)
_,_ = inspect_knee_moment_dataset(torch.squeeze(x),y)

In [ ]:
cnn_model = KneeCNN().to(device)
checkpoint = train_model(cnn_model, train_dataloader, test_dataloader, 
                         num_epochs=10)
plot_loss(checkpoint['train_losses'], checkpoint['test_losses'])

In [ ]:
preds, targets, rmse, r2 = evaluate_model(cnn_model, test_dataloader, 
                                          train_dataset.scaler_y)
print(f'RMSE: {rmse:.4f} Nm')
print(f'R\u00b2:   {r2:.4f}')
prediction_overlay(targets, preds, rmse, r2)

# LOSO Evaluation

### Change Architecture

In [ ]:
experiment_name = "base_"
# CNN
cfg = DatasetConfig(dataset_folder = "ProcessedDataTrimmed")
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeCNN, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    experiment_name = experiment_name
    )

In [ ]:
# TCN
cfg = DatasetConfig(dataset_folder = "ProcessedDataTrimmed")
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeTCN, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    experiment_name = experiment_name
    )

In [ ]:
# LSTM
cfg = DatasetConfig(dataset_folder = "ProcessedDataTrimmed",
                    full_horizon_output = True)
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeLSTM, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    experiment_name = experiment_name
    )

## Hyperparameter tuning

### Increasing hidden layer size (default is 64)

In [ ]:
experiment_name = "layer_"
# CNN, hidden size = 128.
cfg = DatasetConfig(dataset_folder = "ProcessedDataTrimmed")
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeCNN, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    hidden_layer_size = 128,
    experiment_name = experiment_name
    )

In [ ]:
# TCN, hidden size = 128.
cfg = DatasetConfig(dataset_folder = "ProcessedDataTrimmed")
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeTCN, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    hidden_layer_size = 128,
    experiment_name = experiment_name
    )

### Changng window size (default is 100)

In [ ]:
experiment_name = "window_"
# CNN
cfg = DatasetConfig(
    dataset_folder = "ProcessedDataTrimmed",
    window_size = 100
    )
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeCNN, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    hidden_layer_size = 128,
    experiment_name = experiment_name
    )

In [ ]:
# TCN
cfg = DatasetConfig(
    dataset_folder = "ProcessedDataTrimmed",
    window_size = 100
    )
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeTCN, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    hidden_layer_size = 128,
    experiment_name = experiment_name
    )

## Sensor Ablation Studies

In [ ]:
# removing Shank IMU
cfg = DatasetConfig(
    ablated_sensors = ["imu_shank"], 
    dataset_folder = "ProcessedDataTrimmed"
    )
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeTCN, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    experiment_name = "ablation_shank_"
    )

In [ ]:
# removing Shank IMU, knee
cfg = DatasetConfig(
    ablated_sensors = ["imu_shank", "angle", "velocity"], 
    dataset_folder = "ProcessedDataTrimmed"
    )
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeTCN, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    experiment_name = "ablation_shank_knee_"
    )


## Periodic vs non-Periodic

In [ ]:
cfg = DatasetConfig(tasks = PERIODIC_TASK_PREFIXES, 
                    test_tasks = ["sit_to_stand"], 
                    dataset_folder = "ProcessedDataTrimmed")
rmses, r2s, last_model = loso_cross_validation(
    SUBJECTS, KneeTCN, 
    dataset_cfg = cfg, 
    num_epoches = num_epoches,
    experiment_name = "periodic_"
    )